# Lab 1: LLM Data Pipeline with IMDB Reviews and DistilGPT-2

This notebook builds a complete language-modeling data pipeline using long-form movie reviews from the IMDB dataset. Compared with the original version, this lab now uses a different dataset, a lighter GPT-style tokenizer, and a small analysis section to show how token length and block size affect the final training data.

In [5]:
# Optional if packages are not installed:
# !pip3 install datasets transformers torch

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

## 1. Load a realistic text dataset

IMDB reviews are a good fit for an LLM data pipeline because they are natural long-form text rather than short classification headlines. For a lab setting, we use a subset of the training split so the notebook stays fast while still demonstrating the full preprocessing flow.

In [7]:
dataset = load_dataset("imdb", split="train[:5000]")
print(f"Number of reviews loaded: {len(dataset)}")
print(f"Sample review preview: {dataset[0]['text'][:200]}...")

C:\Users\Aditya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aditya\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|████

Number of reviews loaded: 5000
Sample review preview: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...


In [15]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})

## 2. Initialize a GPT-style tokenizer

We switch from `gpt2` to `distilgpt2`. It follows the same causal language-modeling tokenization style, but it is lighter and still makes sense for an LLM preprocessing notebook.

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Pad token id: {tokenizer.pad_token_id}")

C:\Users\Aditya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aditya\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer vocab size: 50257
Pad token id: 50256


## 3. Tokenize the reviews

For language modeling we keep raw token sequences instead of truncating them immediately. That allows us to concatenate text and later build fixed-length training blocks.

In [9]:
def tokenize_function(examples):
    return tokenizer(examples["text"], add_special_tokens=False)

tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

print("First 25 token ids from the first review:")
print(tokenized_ds[0]["input_ids"][:25])

Map: 100%|██████████| 5000/5000 [00:01<00:00, 3983.59 examples/s]

First 25 token ids from the first review:
[40, 26399, 314, 3001, 327, 47269, 20958, 12, 56, 23304, 3913, 422, 616, 2008, 3650, 780, 286, 477, 262, 10386, 326, 11191, 340, 618, 340]


## 4. Analyze token lengths before grouping

This is one of the custom modifications in the lab. We inspect the tokenized review lengths to understand how much text each example contributes before chunking everything into training sequences.

In [10]:
token_lengths = [len(example["input_ids"]) for example in tokenized_ds]

avg_length = sum(token_lengths) / len(token_lengths)
min_length = min(token_lengths)
max_length = max(token_lengths)
total_tokens = sum(token_lengths)

print(f"Average tokenized review length: {avg_length:.2f}")
print(f"Shortest tokenized review: {min_length}")
print(f"Longest tokenized review: {max_length}")
print(f"Total tokens across subset: {total_tokens}")

Average tokenized review length: 296.20
Shortest tokenized review: 12
Longest tokenized review: 2072
Total tokens across subset: 1481017


## 5. Compare block sizes

Instead of only hardcoding one sequence length, we compare a few choices. This makes the notebook more useful as an LLM data pipeline analysis because sequence length directly affects the number of training examples and memory usage.

In [11]:
for candidate_block_size in [64, 128, 256]:
    approx_sequences = total_tokens // candidate_block_size
    print(f"Block size {candidate_block_size}: about {approx_sequences} full training sequences")

Block size 64: about 23140 full training sequences
Block size 128: about 11570 full training sequences
Block size 256: about 5785 full training sequences


## 6. Convert the token stream into fixed-length training blocks

For autoregressive language modeling, we concatenate tokenized text and split it into equal-length chunks. Each chunk becomes one training example.

In [12]:
block_size = 128

def group_texts(examples):
    concatenated_inputs = sum(examples["input_ids"], [])
    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]

    result_input_ids = [
        concatenated_inputs[i:i + block_size]
        for i in range(0, total_len, block_size)
    ]
    result_attention_masks = [[1] * block_size for _ in range(len(result_input_ids))]
    return {
        "input_ids": result_input_ids,
        "attention_mask": result_attention_masks,
    }

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=200)
print(f"Number of final LM training sequences: {len(lm_ds)}")

Map: 100%|██████████| 5000/5000 [00:01<00:00, 2886.81 examples/s]

Number of final LM training sequences: 11556


## 7. Build a PyTorch DataLoader

The DataLoader produces batches ready for a causal language model. We set `labels = input_ids.clone()` because decoder-only transformer models handle the internal shifting during training.

In [13]:
def collate_fn(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    attention_mask = torch.tensor([example["attention_mask"] for example in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone(),
    }

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

## 8. Verify the final batch structure

This confirms that the pipeline is producing tensors with the expected shape for LLM training.

In [14]:
for batch in train_loader:
    print("input_ids shape:", batch["input_ids"].shape)
    print("attention_mask shape:", batch["attention_mask"].shape)
    print("labels shape:", batch["labels"].shape)
    print("Decoded preview from the first sequence in the batch:")
    print(tokenizer.decode(batch["input_ids"][0][:60]))
    break

input_ids shape: torch.Size([8, 128])
attention_mask shape: torch.Size([8, 128])
labels shape: torch.Size([8, 128])
Decoded preview from the first sequence in the batch:
 things followed by lots of walking from place to place and lead to lead.Periodically through out the film various people get undressed and everything has more than a touch of S&M to the proceedings. The violence and fetish material is of the sort to provoke laughter rather than horror or even excitement


## Conclusion

This lab now demonstrates an end-to-end LLM data pipeline on IMDB reviews using the DistilGPT-2 tokenizer. The notebook loads raw long-form text, tokenizes it, analyzes token lengths, compares block sizes, converts the token stream into fixed-length language-modeling examples, and finally batches those examples for PyTorch training.